# Principal Component Analysis (PCA)

## Overview

PCA finds orthogonal axes (principal components) that capture the maximum variance in the data. It is the most widely used dimensionality reduction method and a standard first step before clustering, regression on high-dimensional data, or visualisation.

**Key outputs:**

| Output | Meaning |
|---|---|
| Explained variance ratio | Proportion of total variance each PC captures |
| Loadings | Contribution of each original feature to each PC |
| Scores | Each observation's coordinates in PC space |
| Biplot | Scores and loadings in the same plot |

PCA is a linear method — it captures linear correlations. For non-linear structure use t-SNE or UMAP.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
n = 200
# Correlated ecological features
elevation   = rng.uniform(50, 400, n)
nitrate     = -0.03*elevation + rng.normal(8, 1.5, n)
phosphorus  = 0.6*nitrate + rng.normal(0, 0.8, n)
ph          = 0.002*elevation + rng.normal(6.8, 0.3, n)
richness    = 0.04*elevation - 0.5*nitrate + rng.normal(18, 3, n)
conductance = 0.8*nitrate + rng.normal(0, 1, n)

X = np.column_stack([elevation, nitrate, phosphorus, ph, richness, conductance])
feat_names = ["elevation","nitrate","phosphorus","ph","richness","conductance"]
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)
print(f"Dataset: {X.shape}, correlation range: [{np.corrcoef(X_sc.T)[np.triu_indices(6,1)].min():.2f}, {np.corrcoef(X_sc.T)[np.triu_indices(6,1)].max():.2f}]")

---
## Fitting PCA and Explained Variance

In [ ]:
pca = PCA(random_state=42)
pca.fit(X_sc)
evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].bar(range(1,7), evr*100, color="steelblue", alpha=0.8)
axes[0].plot(range(1,7), cum_evr*100, "o-", color="#e74c3c", lw=2, label="Cumulative")
axes[0].axhline(80, color="grey", linestyle="--", lw=1, label="80% threshold")
axes[0].set_xlabel("Principal Component"); axes[0].set_ylabel("Variance explained (%)")
axes[0].set_title("Scree Plot"); axes[0].legend()
# Kaiser criterion: keep PCs with eigenvalue > 1
eigenvalues = pca.explained_variance_
axes[1].bar(range(1,7), eigenvalues, color="steelblue", alpha=0.8)
axes[1].axhline(1, color="#e74c3c", lw=1.5, linestyle="--", label="Kaiser criterion (eigenvalue=1)")
axes[1].set_xlabel("PC"); axes[1].set_ylabel("Eigenvalue")
axes[1].set_title("Eigenvalues (Kaiser criterion)"); axes[1].legend()
plt.tight_layout(); plt.show()
for i, (ev, cum) in enumerate(zip(evr, cum_evr)):
    print(f"PC{i+1}: {ev:.3f} ({cum:.3f} cumulative)")

---
## Loadings Heatmap

In [ ]:
loadings = pd.DataFrame(pca.components_[:4].T,
                        index=feat_names,
                        columns=[f"PC{i+1}" for i in range(4)])
fig, ax = plt.subplots(figsize=(7,4))
im = ax.imshow(loadings.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax)
ax.set_xticks(range(4)); ax.set_xticklabels(loadings.columns)
ax.set_yticks(range(len(feat_names))); ax.set_yticklabels(feat_names)
for i in range(len(feat_names)):
    for j in range(4):
        ax.text(j, i, f"{loadings.values[i,j]:.2f}", ha="center", va="center",
                fontsize=9, color="white" if abs(loadings.values[i,j])>0.5 else "black")
ax.set_title("PCA Loadings: Contribution of Each Feature to Each PC")
plt.tight_layout(); plt.show()
print("PC1 interpretation: large positive loadings on", loadings["PC1"].abs().nlargest(2).index.tolist())
print("PC2 interpretation: large positive loadings on", loadings["PC2"].abs().nlargest(2).index.tolist())

---
## Biplot: Scores and Loadings Together

In [ ]:
pca2 = PCA(n_components=2, random_state=42)
scores = pca2.fit_transform(X_sc)
loadings2 = pca2.components_.T
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(scores[:,0], scores[:,1], alpha=0.4, s=20, color="steelblue")
scale = 3
for i, feat in enumerate(feat_names):
    ax.arrow(0, 0, loadings2[i,0]*scale, loadings2[i,1]*scale,
             head_width=0.08, head_length=0.05, fc="#e74c3c", ec="#e74c3c", lw=1.5)
    ax.text(loadings2[i,0]*scale*1.15, loadings2[i,1]*scale*1.15,
            feat, fontsize=9, color="#e74c3c", ha="center")
ax.axhline(0, color="grey", lw=0.5); ax.axvline(0, color="grey", lw=0.5)
ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.1%} var)")
ax.set_title("PCA Biplot: Observations (dots) and Feature Loadings (arrows)")
plt.tight_layout(); plt.show()

---
## Choosing Number of Components

In [ ]:
# Three criteria -- should agree on a reasonable range
n_80 = np.argmax(cum_evr >= 0.80) + 1
n_kaiser = (eigenvalues >= 1).sum()
print(f"80% variance threshold: keep {n_80} PCs")
print(f"Kaiser criterion (eigenvalue>=1): keep {n_kaiser} PCs")
print(f"Scree plot elbow: inspect visually (typically 2-3 here)")
# Reconstruction error for different n_components
errors = []
for n in range(1, 7):
    pca_n = PCA(n_components=n, random_state=42)
    X_recon = pca_n.inverse_transform(pca_n.fit_transform(X_sc))
    errors.append(np.mean((X_sc - X_recon)**2))
plt.figure(figsize=(7,4))
plt.plot(range(1,7), errors, "o-", color="steelblue", lw=2)
plt.xlabel("Number of components"); plt.ylabel("Reconstruction MSE")
plt.title("Reconstruction Error vs Components Retained")
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Not standardising before PCA**  
PCA maximises variance. Features on large scales (e.g. elevation in metres) dominate the first PC regardless of their scientific importance. Always apply `StandardScaler` before PCA unless all features are already on comparable scales.

**2. Interpreting PC scores without examining loadings**  
A PC score is meaningless without knowing which features drive it. Always examine the loadings heatmap and interpret each retained PC in terms of its dominant features before drawing ecological or scientific conclusions.

**3. Choosing the number of components by the 80% threshold alone**  
The 80% threshold is a rule of thumb, not a law. Combine it with the Kaiser criterion (eigenvalue > 1) and scree plot elbow. If they disagree, use domain knowledge and reconstruction error to guide the decision.

**4. Using PCA scores as predictors without accounting for the train/test split**  
Fit PCA on training data only and apply `transform()` to test data. Fitting on all data before splitting leaks test set variance structure into the components used for training.

**5. Expecting PCA to reveal non-linear structure**  
PCA finds linear combinations of features. Clusters or manifolds that are non-linearly arranged are invisible to PCA but visible in t-SNE or UMAP. Use PCA as a first pass; follow with non-linear methods if the scatterplot shows unexplained structure.
---
*python_methods_library - Samantha McGarrigle*